In [1]:
!pip install ultralytics # installing ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.6 MB/s eta 0:00:00


In [8]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [10]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt")

# Modified


In [11]:
import os
import numpy as np
import pandas as pd
def process_test_sequence_fixed(folderN,folder, det_conf=0.25, closed_thr=0.5, open_thr=0.5,
                                min_blink_frames=3, gap_merge=2):
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.jpg', '.png', '.jpeg'))])

    blink_results = []
    state_seq = []
    state_seq = [0] + state_seq
    closed_conf_seq = []
    closed_conf_seq = [0] + closed_conf_seq

    prev_state = 0  # assume open at start

    for f in files:
        img_path = os.path.join(folder, f)
        res = model.predict(img_path, conf=det_conf, verbose=False)

        closed_conf = 0.0
        open_conf = 0.0


        if res and len(res) > 0 and res[0].boxes is not None and len(res[0].boxes) > 0:
            for box in res[0].boxes:
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                class_name = model.names[cls_id]

                if class_name == 'closed eyes':
                    closed_conf = max(closed_conf, conf)
                elif class_name == 'opened eyes':
                    open_conf = max(open_conf, conf)

        closed_conf_seq.append(closed_conf)
        frame_num = int(f.split('_')[-1].split('.')[0])
        index = len(closed_conf_seq) - 1


        # Stable frame-state decision
        if closed_conf >= closed_thr :# closed_conf >= open_conf cause model is bias towerd open
            state = 1
        elif open_conf >= open_thr and open_conf > closed_conf:
            state = 0
        else:
            state = 0 # carry previous state if uncertain / no detection

        state_seq.append(state)
        prev_state = state

    state_seq = np.array(state_seq, dtype=int)
    closed_conf_seq = np.array(closed_conf_seq, dtype=float)
    print("closed conf min/max:", min(closed_conf_seq), max(closed_conf_seq))
    print("first 50 closed confs:", closed_conf_seq[:50])#good
    # Remove tiny glitches: 0,1,0 or 1,0,1
    for i in range(1, len(state_seq) - 1):
        if state_seq[i - 1] == state_seq[i + 1]:
            state_seq[i] = state_seq[i - 1]

    # Extract closed runs
    raw_blinks = []
    in_blink = False

    si = -1
    print("num closed frames:", int(np.sum(state_seq)))

    for t in range(len(state_seq)):
        if not in_blink and state_seq[t] == 1:
            si = t
            in_blink = True
        elif in_blink and state_seq[t] == 0:
            ei = t - 1
            if ei >= si:
                raw_blinks.append((si, ei))
            in_blink = False #set them again
            si = -1

    if in_blink and si != -1:
        raw_blinks.append((si, len(state_seq) - 1))

    # Merge blink fragments separated by tiny open gaps
    merged = []
    for seg in raw_blinks:
        if not merged:
            merged.append(seg)
        else:
            prev_si, prev_ei = merged[-1]
            cur_si, cur_ei = seg
            if cur_si - prev_ei - 1 <= gap_merge:
                merged[-1] = (prev_si, cur_ei)
            else:
                merged.append(seg)

    print("Raw closed runs:")
    for si, ei in merged:
      print((si, ei), "duration =", ei - si + 1)


    # Keep only realistic blink durations AFTER trimming
    blinks = []
    for si, ei in merged:
        duration = ei - si + 1
        if min_blink_frames <= duration :
            blinks.append((si, ei))

    print("State sequence:")
    print(state_seq.tolist())
    print("\nDetected blinks:", blinks)

    if not blinks:
        print("No blink detected.")
        return

    maxi = 0
    for blink_idx, (si, ei) in enumerate(blinks):
        duration = ei - si + 1
        maxi = max(maxi, duration)

        # peak inside blink from closed confidence
        seg = closed_conf_seq[si:ei+1]
        bi = si + int(np.argmax(seg))

        baseline = min(closed_conf_seq[si], closed_conf_seq[ei])
        amplitude = closed_conf_seq[bi] - baseline

        if ei > bi:
            velocity = (closed_conf_seq[bi] - closed_conf_seq[ei]) / (ei - bi)
        else:
            velocity = 0.0#to avoid runtime error

        frequency = 100 * ((blink_idx + 1) / (ei + 1))#consider changing it
        print("start value:", closed_conf_seq[si])
        print("end value:", closed_conf_seq[ei])
        print(f"\n--- Blink {blink_idx + 1} ---")
        print(f"Interval B_i: [{si}, {ei}]")
        print(f"Duration: {duration} frames")
        print(f"baseline: {baseline} ")
        print(f"peak: {bi} ")
        print(f"Amplitude: {amplitude:.4f}")
        print(f"Reopening Velocity: {velocity:.4f}")
        print(f"Blink Frequency: {frequency:.2f} per 100 frames")
        print("first 50 closed confs:", closed_conf_seq[:50])

        blink_results.append({
            "Blink_ID": blink_idx + 1,
            "Start_Frame": si,
            "End_Frame": ei,
            "Duration": duration,
            "Amplitude": amplitude,
            "Velocity": velocity,
            "Frequency": frequency
        })
        df = pd.DataFrame(blink_results)
        base_path = '/content/drive/MyDrive/GP/upated_CSV/training'



        save_dir = os.path.join(base_path, label)
        os.makedirs(save_dir, exist_ok=True)

        csv_path = os.path.join(save_dir, f"{folderN}.csv")
        df.to_csv(csv_path, index=False)

        print("Saved:", csv_path)


**run all files here**

In [ ]:
import os

folder_path = '/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Drowsy' #<-- Change this to your folder
image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff")

folders_with_images = []

# Step 1: collect folders
for root, dirs, files in os.walk(folder_path):
    if any(file.lower().endswith(image_extensions) for file in files):
        folders_with_images.append(root)

# Step 2: clean list
folders_with_images = sorted(set(folders_with_images))

print("Found folders:", len(folders_with_images))

# Step 3: process
for folder in folders_with_images:
    print("\nProcessing:", folder)

    folder_name = os.path.basename(folder)

    process_test_sequence_fixed(
        folder_name,folder,   # ✅ pass FULL PATH only
        det_conf=0.44,
        closed_thr=0.25,
        open_thr=0.25
    )

Found folders: 14

Processing: /content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Drowsy/D001_20260415_074741_frames
closed conf min/max: 0.0 0.8733960390090942
first 50 closed confs: [          0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0
           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0     0.83808      0.8288           0           0           0           0           0]
num closed frames: 3748
Raw closed runs:
(43, 44) duration = 2
(110, 251) duration = 142
(266, 270) duration = 5
(314, 495) duration = 182
(510, 518) duration = 9
(612, 615) duration = 4
(650, 670) du

error: OpenCV(4.13.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1310: error: (-215:Assertion failed) !buf.empty() in function 'imdecode_'


In [14]:
import os
# is stopped use this
folder_path = '/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy'
image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff")

start_folder = "/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D008_20260511_142830_frames"

folders_with_images = []

# Step 1: collect folders
for root, dirs, files in os.walk(folder_path):
    if any(file.lower().endswith(image_extensions) for file in files):
        folders_with_images.append(root)

folders_with_images = sorted(set(folders_with_images))

for f in folders_with_images:
    print(repr(f))

# Step 2: find where to start
if start_folder not in folders_with_images:
    print("⚠️ Start folder not found!")
else:
    start_idx = folders_with_images.index(start_folder)

    # Step 3: process from that point onward
    for folder in folders_with_images[start_idx:]:
        print("\nProcessing:", folder)

        folder_name = os.path.basename(folder)

        process_test_sequence_fixed(
            folder_name,folder,
            det_conf=0.44,
            closed_thr=0.25,
            open_thr=0.25
        )

'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D001'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D002_20260512_231154_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D004'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D005'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D006'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D007'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D008_20260511_142830_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D009'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D010'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D011'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D012'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D013'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D014'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/D015'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/Drowsy/

error: OpenCV(4.13.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1310: error: (-215:Assertion failed) !buf.empty() in function 'imdecode_'
